# STAR Project – ANOVA

**Question:** is the average math score different between small, regular, and regular + aide classes?

**ANOVA** compares the averages of more than two groups. Small p (< 0.05) = the groups really differ.

> 💡 **בעברית:** ANOVA משווה ממוצעים של יותר משתי קבוצות. p קטן (< 0.05) = הקבוצות באמת שונות.

## 1. Setup

Groups = `first_classtype` (like in notebook 01). **Grade 5 is not used** (decision with our advisor).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_spss("../data/raw/STAR_Students.sav")
df = df.copy()

k3 = ["gk", "g1", "g2", "g3"]
grades = ["gk", "g1", "g2", "g3", "g4", "g6", "g7", "g8"]          # no grade 5
grade_labels = ["K", "1", "2", "3", "4", "6", "7", "8"]

df["first_classtype"] = df[[f"{g}classtype" for g in k3]].astype(object).bfill(axis=1).iloc[:, 0]

order = ["SMALL CLASS", "REGULAR CLASS", "REGULAR + AIDE CLASS"]
short = ["Small", "Regular", "Regular + aide"]
colors = ["tab:green", "tab:orange", "tab:blue"]

## 2. ANOVA in kindergarten

Mean of each group with a 95% confidence interval, and the ANOVA result.

In [ ]:
k = df[["first_classtype", "gktmathss"]].dropna()
groups = [k.loc[k["first_classtype"] == c, "gktmathss"] for c in order]

summary = k.groupby("first_classtype")["gktmathss"].agg(["count", "mean", "std"]).reindex(order)
err = 1.96 * summary["std"] / np.sqrt(summary["count"])      # 95% CI half-width
display(summary.round(1))

f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA:  F = {f_stat:.2f},  p = {p_value:.2g}")

for i, c in enumerate(colors):
    plt.errorbar(i, summary["mean"].iloc[i], yerr=err.iloc[i], fmt="o", capsize=6, color=c, markersize=8)
plt.xticks(range(3), short)
plt.xlim(-0.5, 2.5)
plt.ylabel("Math score")
plt.title(f"Mean math score in K (95% CI)\nANOVA: F = {f_stat:.1f}, p = {p_value:.1g}")
plt.savefig("../figures/anova_k.png", dpi=150, bbox_inches="tight")
plt.show()

**What we see:** _fill in after running_

## 3. ANOVA in every grade

The same test in every grade (no grade 5). Each panel has its own y-axis, so compare the groups **inside** a panel.

In [ ]:
rows = []
for g, lab in zip(grades, grade_labels):
    d = df[["first_classtype", f"{g}tmathss"]].dropna()
    f, p = stats.f_oneway(*[d.loc[d["first_classtype"] == c, f"{g}tmathss"] for c in order])
    rows.append({"grade": lab, "students": len(d), "F": f, "p-value": p})

anova_by_grade = pd.DataFrame(rows).set_index("grade")
anova_by_grade["significant"] = np.where(anova_by_grade["p-value"] < 0.05, "yes", "no")
display(anova_by_grade.round(4))

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, g, lab in zip(axes.flat, grades, grade_labels):
    s = df.groupby("first_classtype")[f"{g}tmathss"].agg(["count", "mean", "std"]).reindex(order)
    e = 1.96 * s["std"] / np.sqrt(s["count"])
    for i, c in enumerate(colors):
        ax.errorbar(i, s["mean"].iloc[i], yerr=e.iloc[i], fmt="o", capsize=5, color=c)
    ax.set_xticks(range(3), ["S", "R", "RA"])
    ax.set_xlim(-0.5, 2.5)
    ax.set_title(f"Grade {lab}  (p = {anova_by_grade.loc[lab, 'p-value']:.2g})")

fig.suptitle("Mean math score by class type (95% CI)   S = small, R = regular, RA = regular + aide")
plt.tight_layout()
plt.savefig("../figures/anova_by_grade.png", dpi=150, bbox_inches="tight")
plt.show()

**What we see:** _fill in after running_

**Limitations:** this simple ANOVA ignores schools (randomization was inside schools) and dropout. It is a first look only.

> 💡 **בעברית:** ה-ANOVA הזו לא מתחשבת בבתי ספר ובנשירה. זה מבט ראשון בלבד.